In [1]:
import sleap
import numpy as np

# This prevents TensorFlow from allocating all the GPU memory, which leads to issues on
# some GPUs/platforms:
# sleap.disable_preallocation()

# This would hide GPUs from the TensorFlow altogether:
# sleap.use_cpu_onl
# Print some info:
sleap.versions()
sleap.system_summary()

SLEAP: 1.4.1
TensorFlow: 2.7.0
Numpy: 1.21.5
Python: 3.7.12
OS: Linux-6.8.0-51-generic-x86_64-with-debian-trixie-sid
GPUs: 2/2 available
  Device: /physical_device:GPU:0
         Available: True
       Initialized: False
     Memory growth: None
  Device: /physical_device:GPU:1
         Available: True
       Initialized: False
     Memory growth: None


In [2]:
import os
import numpy as np
import sys
sys.path.append('/home/mingxiao/Desktop/jelly-sleap/sleap')
sys.path.append('/home/mingxiao/jelly-sleap/sleap')
sys.path.append('/home/mingxiao/')

In [24]:
animal_id = 4
original_labels_dir = f'/home/mingxiao/Desktop/jellyfish/label/animal_{animal_id}_labels.v001.slp'
# original_labels_dir = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v5.slp'
original_labels = sleap.load_file(original_labels_dir)

In [25]:
original_labels

Labels(labeled_frames=3001, videos=1, skeletons=1, tracks=0)

In [26]:
new_labels_dir = f'/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_{animal_id}_v1.slp'
new_labels = sleap.load_file(new_labels_dir)

In [27]:
tb_cnt = len(original_labels.skeletons[0].nodes) - 1
print(f'tb_cnt: {tb_cnt}')
new_skeletons = []
for i in range(tb_cnt):
    new_skeleton = sleap.Skeleton(name=f'TB{i+1}')
    new_skeleton.add_node(f'tb{i+1}_node')
    new_skeletons.append(new_skeleton)
new_skeletons

tb_cnt: 16


[Skeleton(name='TB1', description='None', nodes=['tb1_node'], edges=[], symmetries=[]),
 Skeleton(name='TB2', description='None', nodes=['tb2_node'], edges=[], symmetries=[]),
 Skeleton(name='TB3', description='None', nodes=['tb3_node'], edges=[], symmetries=[]),
 Skeleton(name='TB4', description='None', nodes=['tb4_node'], edges=[], symmetries=[]),
 Skeleton(name='TB5', description='None', nodes=['tb5_node'], edges=[], symmetries=[]),
 Skeleton(name='TB6', description='None', nodes=['tb6_node'], edges=[], symmetries=[]),
 Skeleton(name='TB7', description='None', nodes=['tb7_node'], edges=[], symmetries=[]),
 Skeleton(name='TB8', description='None', nodes=['tb8_node'], edges=[], symmetries=[]),
 Skeleton(name='TB9', description='None', nodes=['tb9_node'], edges=[], symmetries=[]),
 Skeleton(name='TB10', description='None', nodes=['tb10_node'], edges=[], symmetries=[]),
 Skeleton(name='TB11', description='None', nodes=['tb11_node'], edges=[], symmetries=[]),
 Skeleton(name='TB12', descr

In [28]:
new_labels.skeletons = new_skeletons

In [29]:
new_labeled_frames = []
for lf in original_labels.labeled_frames:
    # generate new instances 
    # new_instances = [None for _ in range(tb_cnt)]
    new_instances = []
    labeled_inst = None
    for inst in lf.instances: # each frame has at most 2 instances: labeled and predicted
        if isinstance(inst, sleap.Instance) and not isinstance(inst, sleap.PredictedInstance):
            labeled_inst = inst
            break
    if labeled_inst is None:
        continue
    for old_node, old_point in zip(lf.instances[0].nodes, lf.instances[0].points):
        if old_node.name.lower() == 'mouth':
            continue
        tb_idx = int(old_node.name[2:])
        tb_skeleton = new_skeletons[tb_idx - 1]
        point_dict = {f'tb{tb_idx}_node': sleap.instance.Point(x=old_point.x, y=old_point.y)}
        tb_instance = sleap.Instance(skeleton=tb_skeleton, points=point_dict, frame=lf)
        # new_instances[tb_idx - 1] = tb_instance
        new_instances.append(tb_instance)
    # print the indices where new_instances are None
    if None in new_instances:
        print(f'None indices: {np.where(np.array(new_instances) == None)[0]}')
    assert None not in new_instances, f'some instances are None'
    new_lf = sleap.LabeledFrame(video=new_labels.video, frame_idx=lf.frame_idx, instances=new_instances)
    new_labeled_frames.append(new_lf)

In [30]:
new_labels.labeled_frames = new_labeled_frames
new_labels

Labels(labeled_frames=869, videos=1, skeletons=16, tracks=0)

In [31]:
new_suggestions = []
for suggestion in original_labels.suggestions:
    new_suggestion = sleap.gui.suggestions.SuggestionFrame(video=new_labels.video, frame_idx=suggestion.frame_idx, group=suggestion.group)
    new_suggestions.append(new_suggestion)
new_labels.suggestions = new_suggestions

In [32]:
new_labels

Labels(labeled_frames=869, videos=1, skeletons=16, tracks=0)

In [33]:
new_labels.save(new_labels_dir)